# ProtSpace: Interactive Protein Embedding Visualization

Interactive visualization of high-dimensional protein embeddings in 2D/3D space. Supports multiple dimensionality reduction methods (PCA, UMAP, t-SNE, PaCMAP, MDS, LocalMAP) with annotation-based coloring and integrated structure viewing.

📚 [GitHub](https://github.com/tsenoner/protspace) • [Manuscript](https://www.sciencedirect.com/science/article/pii/S0022283625000063?via%3Dihub)


In [ ]:
# @title Install Dependencies and Import Libraries (~1min)
%%capture
!pip install -q "protspace @ git+https://github.com/tsenoner/protspace.git@stage"

import gzip
import io
import os
import re
import subprocess
from pathlib import Path

import h5py
import ipywidgets as widgets
from google.colab import drive, files
from IPython.display import clear_output, display
from ipywidgets import (
    HTML,
    BoundedIntText,
    Button,
    Checkbox,
    Dropdown,
    FileUpload,
    FloatSlider,
    GridBox,
    HBox,
    IntProgress,
    IntSlider,
    Output,
    SelectMultiple,
    Tab,
    ToggleButton,
    VBox,
)

zsh:1: command not found: pip


# Try It Now — Example Datasets

New to ProtSpace? Start here! Download a precomputed example dataset and jump straight to generating a visualization bundle. You can always upload your own data in the next cell.


In [2]:
# @title 🎯 Example Datasets — Try It Now {display-mode: "form"}
# @markdown Download a precomputed example dataset to try ProtSpace immediately.

import os
import urllib.request
from pathlib import Path

import h5py
import ipywidgets as widgets
from IPython.display import clear_output, display

# Example datasets hosted on GitHub releases
_EXAMPLES = {
    "Phosphatase (832 proteins, 1.9 MB)": "phosphatase.h5",
    "3FTx Toxins (1,428 proteins, 3.4 MB)": "3ftx_prott5.h5",
    "Pfam Families (2,222 proteins, 5.1 MB)": "pfam_PF00061_PF00067_PF00077.h5",
    "Subcellular Localization (6.4 MB)": "deeploc_test_set_prott5.h5",
}
_RELEASE_URL = "https://github.com/tsenoner/protspace/releases/download/v3.3.1/"

embedding_file = None  # global, used by later cells

example_dropdown = widgets.Dropdown(
    options=list(_EXAMPLES.keys()),
    value=list(_EXAMPLES.keys())[0],
    description="Dataset:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="60%"),
)
download_btn = widgets.Button(
    description="Download Example", button_style="success", icon="download"
)
example_output = widgets.Output()


def _download_example(_b):
    global embedding_file
    with example_output:
        clear_output()
        filename = _EXAMPLES[example_dropdown.value]
        url = _RELEASE_URL + filename
        print(f"⏳ Downloading {filename}...")
        try:
            urllib.request.urlretrieve(url, filename)
        except Exception as e:
            print(f"❌ Download failed: {e}")
            print(f"   URL: {url}")
            print("   Check your internet connection and try again.")
            return

        # Validate
        try:
            with h5py.File(filename, "r") as f:
                keys = list(f.keys())
                if not keys:
                    print("❌ Downloaded file is empty.")
                    return
                first_key = keys[0]
                item = f[first_key]
                if hasattr(item, "shape"):
                    shape = item.shape
                else:
                    print(f"❌ Unexpected HDF5 structure (groups instead of datasets).")
                    return
            size_mb = os.path.getsize(filename) / (1024 * 1024)
            print(f"✅ {filename} ready ({len(keys)} proteins, {size_mb:.1f} MB)")
            embedding_file = filename
        except Exception as e:
            print(f"❌ Validation failed: {e}")


download_btn.on_click(_download_example)

print("=" * 60)
print("🎯 EXAMPLE DATASETS")
print("=" * 60)
print("\nSelect a dataset and click Download, then skip to the Generate cell.")
print("Or skip this cell and upload your own data below.\n")
display(
    widgets.VBox(
        [
            widgets.HBox([example_dropdown, download_btn]),
            example_output,
        ]
    )
)

🎯 EXAMPLE DATASETS

Select a dataset and click Download, then skip to the Generate cell.
Or skip this cell and upload your own data below.



# Upload Your Own Embedding Data

Skip this section if you're using an example dataset from above.

## How to Get Protein Embeddings

1.  **From UniProt:**
    - Go to the [UniProt website](https://www.uniprot.org/).
    - Use UniProt search syntax (e.g., `(ft_domain:phosphatase) AND (reviewed:true)`).
    - Click **"Customize"** → Select **"Embeddings"** to generate ProtT5 embeddings.
    - Download the results from your [Jobs Dashboard](https://www.uniprot.org/tool-dashboard).

2.  **From your own FASTA file:**
    - Generate embeddings using the dedicated notebook: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsenoner/protspace/blob/main/notebooks/ClickThrough_GenerateEmbeddings.ipynb)


In [ ]:
# @title 📂 Upload Embedding File {display-mode: "form"}
# @markdown Upload your own embedding file (.h5, .hdf5, or .gz).

import gzip
import io
import os
from pathlib import Path

import h5py
import ipywidgets as widgets
from google.colab import drive, files
from IPython.display import clear_output, display


def process_and_validate(filename, file_content, output_widget):
    """Write file to disk, validate as HDF5, return output path or None."""
    filepath = Path(filename)
    output_name = None

    try:
        if filepath.suffix == ".gz":
            output_name = filepath.stem + ".h5"
            with gzip.open(io.BytesIO(file_content), "rb") as gz_f:
                decompressed_data = gz_f.read()
            with open(output_name, "wb") as f:
                f.write(decompressed_data)
        elif filepath.suffix in (".h5", ".hdf5", ""):
            output_name = (
                filename
                if filepath.suffix in (".h5", ".hdf5")
                else filepath.name + ".h5"
            )
            with open(output_name, "wb") as f:
                f.write(file_content)
        else:
            with output_widget:
                print(
                    f"❌ Unsupported format '{filepath.suffix}'. Use .h5, .hdf5, or .gz."
                )
            return None

        # Validate HDF5
        with h5py.File(output_name, "r") as f:
            keys = list(f.keys())
            if not keys:
                raise ValueError("HDF5 file is empty (no datasets found)")

            first_key = keys[0]
            item = f[first_key]

            # Handle groups: look for datasets inside
            if isinstance(item, h5py.Group):
                sub_keys = list(item.keys())
                if not sub_keys:
                    raise ValueError(f"HDF5 group '{first_key}' is empty")
                item = item[sub_keys[0]]
                first_key = sub_keys[0]

            emb = item[:]

            # Handle 2D embeddings like (1, 1024)
            if emb.ndim == 2 and emb.shape[0] == 1:
                emb = emb.squeeze(axis=0)
            elif emb.ndim > 1:
                raise ValueError(
                    f"Found per-residue embeddings (shape {emb.shape}). "
                    f"ProtSpace requires per-protein embeddings (1D vectors). "
                    f"Use mean-pooling or CLS token extraction first."
                )

            if emb.ndim != 1 or emb.shape[0] == 0:
                raise ValueError(f"Expected 1-D embedding, got shape {emb.shape}")

            n_proteins = len(keys)

        with output_widget:
            print(f"✅ Embedding file ready: {output_name}")
            print(
                f"   Proteins: {n_proteins}, Embedding dim: {emb.shape[0]}, Dtype: {emb.dtype}"
            )
            if n_proteins < 10:
                print(f"   ⚠️ Only {n_proteins} proteins — results may be limited.")
        return output_name

    except Exception as e:
        with output_widget:
            print(f"❌ Failed to process '{filename}': {e}")
        if output_name and os.path.exists(output_name):
            os.remove(output_name)
        return None


# ---------------------------------------------------------------------------
# Upload methods
# ---------------------------------------------------------------------------

# embedding_file may already be set by the example data cell
if "embedding_file" not in dir():
    embedding_file = None


def method_google_drive(output_widget):
    """Large files from Google Drive."""
    global embedding_file

    with output_widget:
        print("📁 Google Drive Upload")
        print("=" * 60)
        print("Mounting Google Drive...")
        try:
            drive.mount("/content/drive", force_remount=False)
            print("✅ Google Drive mounted.")
        except Exception:
            print("⚠️ Google Drive already mounted.")

        drive_path = "/content/drive/MyDrive"
        embedding_files = []
        try:
            for root, _dirs, files_list in os.walk(drive_path):
                for f in files_list:
                    if Path(f).suffix in (".h5", ".hdf5", ".gz"):
                        embedding_files.append(
                            os.path.relpath(os.path.join(root, f), drive_path)
                        )
        except Exception as e:
            print(f"⚠️ Could not list Drive files: {e}")

        if not embedding_files:
            print("\n⚠️ No embedding files (.h5, .hdf5, .gz) found in your Drive.")
            return

        file_options = [
            (
                f"{f} ({os.path.getsize(os.path.join(drive_path, f)) / (1024 * 1024):.1f} MB)",
                f,
            )
            for f in sorted(
                embedding_files,
                key=lambda x: os.path.getmtime(os.path.join(drive_path, x)),
                reverse=True,
            )
        ]

        file_dropdown = widgets.Dropdown(
            options=file_options,
            description="Select File:",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="80%"),
        )
        status_output = widgets.Output()

        def on_process_click(_b):
            global embedding_file
            with status_output:
                clear_output()
            filename = file_dropdown.value
            if not filename:
                with status_output:
                    print("❌ No file selected.")
                return
            file_path = (
                filename
                if filename.startswith("/content/drive")
                else f"/content/drive/MyDrive/{filename}"
            )
            with open(file_path, "rb") as f:
                data = f.read()
            embedding_file = process_and_validate(
                os.path.basename(filename), data, status_output
            )

        process_button = widgets.Button(
            description="Process Selected File",
            button_style="success",
            icon="check",
        )
        process_button.on_click(on_process_click)

        print("\n📝 Select your embedding file:")
        display(
            widgets.VBox(
                [
                    widgets.HBox([file_dropdown, process_button]),
                    status_output,
                ]
            )
        )


def method_colab_upload(output_widget):
    """Colab native file dialog."""
    global embedding_file

    with output_widget:
        print("📁 Colab Native Upload")
        print("⏳ A file dialog will open — select your embedding file.")

    uploaded = files.upload()
    if len(uploaded) != 1:
        with output_widget:
            print("❌ Please upload exactly one embedding file.")
        embedding_file = None
        return

    filename = list(uploaded.keys())[0]
    with output_widget:
        print(f"Processing: {filename}")
    embedding_file = process_and_validate(filename, uploaded[filename], output_widget)


# ---------------------------------------------------------------------------
# Main interface
# ---------------------------------------------------------------------------

_METHODS = {1: method_colab_upload, 2: method_google_drive}

method_selector = widgets.Dropdown(
    options=[
        ("📤 Colab Native Upload (Reliable)", 1),
        ("🚀 Google Drive (Recommended for large files)", 2),
    ],
    value=1,
    description="Method:",
    style={"description_width": "initial"},
)

start_button = widgets.Button(
    description="Start Upload", button_style="success", icon="upload"
)
output_area = widgets.Output()


def on_start_click(_b):
    with output_area:
        clear_output()
    _METHODS[method_selector.value](output_area)


start_button.on_click(on_start_click)

# Check if example data was already loaded
if embedding_file:
    print(f"✅ Using example dataset: {embedding_file}")
    print("You can skip this cell and proceed to Generate.\n")

print("=" * 60)
print("📂 UPLOAD YOUR OWN EMBEDDING FILE")
print("=" * 60)
print("\n📌 Choose your upload method:")
print("   • Colab Native: Reliable for various file sizes.")
print("   • Google Drive: Recommended for large files.\n")

display(widgets.VBox([method_selector, start_button]))
display(output_area)

In [ ]:
# @title 🚀 Generate ProtSpace Parquet Bundle {display-mode: "form"}
# @markdown Click "Generate Bundle" for defaults, or expand Advanced Options to customize.

import os
import re
import subprocess
from pathlib import Path

import ipywidgets as widgets
from IPython.display import clear_output, display
from ipywidgets import (
    HTML,
    Accordion,
    BoundedIntText,
    Button,
    Checkbox,
    Dropdown,
    FileUpload,
    FloatSlider,
    GridBox,
    HBox,
    IntProgress,
    IntSlider,
    Output,
    ToggleButton,
    VBox,
)

# Annotation categories matching the CLI
ANNOTATIONS = {
    "UniProt": [
        "annotation_score",
        "cc_subcellular_location",
        "ec",
        "fragment",
        "go_bp",
        "go_cc",
        "go_mf",
        "keyword",
        "length",
        "protein_existence",
        "protein_families",
        "reviewed",
        "xref_pdb",
    ],
    "InterPro": [
        "cath",
        "cdd",
        "panther",
        "pfam",
        "prints",
        "prosite",
        "signal_peptide",
        "smart",
        "superfamily",
    ],
    "Taxonomy": [
        "root",
        "domain",
        "kingdom",
        "phylum",
        "class",
        "order",
        "family",
        "genus",
        "species",
    ],
}

METHODS = ["PCA", "UMAP", "t-SNE", "MDS", "PaCMAP", "LocalMAP"]

METHOD_TOOLTIPS = {
    "PCA": "Principal Component Analysis (fast, linear)",
    "UMAP": "Uniform Manifold Approximation (local + global structure)",
    "t-SNE": "t-SNE (captures local structure, slower)",
    "MDS": "Multidimensional Scaling (preserves distances)",
    "PaCMAP": "Pairwise Controlled Manifold Approximation (local + global)",
    "LocalMAP": "Locally-aware Manifold Projection (local structure)",
}

# Parameter groups: (label, {param: (min, max, default)}, {methods that use it})
PARAM_GROUPS = [
    (
        "UMAP / PaCMAP / LocalMAP",
        {"n_neighbors": (5, 200, 15)},
        {"UMAP", "PaCMAP", "LocalMAP"},
    ),
    ("UMAP", {"min_dist": (0.0, 1.0, 0.1)}, {"UMAP"}),
    (
        "PaCMAP / LocalMAP",
        {"mn_ratio": (0.1, 1.0, 0.5), "fp_ratio": (1.0, 5.0, 2.0)},
        {"PaCMAP", "LocalMAP"},
    ),
    (
        "t-SNE",
        {"perplexity": (5, 50, 30), "learning_rate": (10, 1000, 200)},
        {"t-SNE"},
    ),
    (
        "MDS",
        {"n_init": (1, 10, 4), "max_iter": (50, 1000, 300), "eps": (0.001, 0.1, 0.001)},
        {"MDS"},
    ),
]

_STOCHASTIC_METHODS = {"UMAP", "t-SNE", "MDS", "PaCMAP", "LocalMAP"}

_GROUP_BOX_LAYOUT = {
    "border": "1px solid #e0e0e0",
    "padding": "8px 12px",
    "margin": "0",
    "overflow": "hidden",
}

ANNOTATION_DEFAULTS = {
    "ec",
    "keyword",
    "length",
    "protein_families",
    "reviewed",
}

# Annotation presets for the dropdown
_ANNOTATION_PRESETS = {
    "Default (recommended)": "default",
    "All annotations": "all",
    "None (skip annotations)": "none",
    "Custom...": "custom",
}

_PROGRESS_TASKS = {
    "UniProt annotations": ("uniprot", "🔍 Fetching UniProt Annotations"),
    "taxonomy annotations": ("taxonomy", "🌿 Fetching Taxonomy Annotations"),
    "InterPro annotations": ("interpro", "🧬 Fetching InterPro Annotations"),
}

_SKIP_PATTERNS = [
    "Unable to register cu",
    "WARNING: All log messages",
    "E0000",
    "W0000",
    "This TensorFlow binary",
    "Creating directory",
    "Welcome to Bioservices",
    "It looks like you do not have",
    "We are creating one with default",
    "Done",
    "To enable the following instructions",
]


class ProtSpaceConfigWidget:
    def __init__(self):
        self._build_widgets()
        self._build_layout()
        self._on_method_change()

    def _build_widgets(self):
        # --- Run button (top-level, always visible) ---
        self.run_button = Button(
            description="🚀 Generate Bundle",
            button_style="primary",
            layout={"width": "220px", "height": "40px"},
        )
        self.run_button.on_click(self._on_run)
        self.output = Output()

        # --- Annotation preset dropdown ---
        self.annotation_preset = Dropdown(
            options=list(_ANNOTATION_PRESETS.keys()),
            value="Default (recommended)",
            description="Annotations:",
            style={"description_width": "initial"},
        )
        self.annotation_preset.observe(self._on_preset_change, names="value")

        # Annotation checkboxes (hidden by default, shown when "Custom..." selected)
        self.annotation_checkboxes = {}
        for category, names in ANNOTATIONS.items():
            self.annotation_checkboxes[category] = {}
            for name in names:
                self.annotation_checkboxes[category][name] = Checkbox(
                    value=name in ANNOTATION_DEFAULTS,
                    description=name,
                    indent=False,
                    layout={"width": "auto"},
                )

        # Per-group All/Clear buttons
        self.group_buttons = {}
        for category in ANNOTATIONS:
            all_btn = Button(description="All", layout={"width": "60px"})
            clear_btn = Button(description="Clear", layout={"width": "60px"})
            all_btn.on_click(lambda _b, cat=category: self._select_group(cat, True))
            clear_btn.on_click(lambda _b, cat=category: self._select_group(cat, False))
            self.group_buttons[category] = (all_btn, clear_btn)

        # CSV metadata upload
        self.csv_upload = FileUpload(
            accept=".csv,.tsv", multiple=False, description="Choose CSV"
        )

        # Method toggle buttons
        self.method_checkboxes = {}
        for method in METHODS:
            selected = method in {"PCA", "UMAP"}
            self.method_checkboxes[method] = ToggleButton(
                value=selected,
                description=method,
                tooltip=METHOD_TOOLTIPS.get(method, ""),
                button_style="info" if selected else "",
                layout={"width": "auto", "height": "32px"},
            )

        _slider_style = {"description_width": "110px"}
        _slider_layout = {"width": "auto"}

        self.param_widgets = {}
        self.param_group_widgets = []

        _general_style = {"description_width": "initial"}
        metric_widget = Dropdown(
            options=["euclidean", "cosine"],
            value="euclidean",
            description="Metric (UMAP, t-SNE, MDS):",
            style=_general_style,
            layout=_slider_layout,
        )
        self.param_widgets["metric"] = metric_widget

        random_state_widget = BoundedIntText(
            value=42,
            min=0,
            max=99999,
            description="Random seed (all except PCA):",
            style=_general_style,
            layout=_slider_layout,
        )
        self.param_widgets["random_state"] = random_state_widget

        general_box = VBox(
            [
                HTML("<b style='font-size:0.9em'>General</b>"),
                metric_widget,
                random_state_widget,
            ],
            layout=_GROUP_BOX_LAYOUT,
        )
        self.param_group_widgets.append((general_box, _STOCHASTIC_METHODS))

        for group_label, params, required_methods in PARAM_GROUPS:
            group_sliders = []
            for pname, (lo, hi, default) in params.items():
                slider_cls = IntSlider if isinstance(default, int) else FloatSlider
                kw = (
                    {"step": 0.1 if (hi - lo) >= 0.5 else 0.001}
                    if slider_cls is FloatSlider
                    else {}
                )
                slider = slider_cls(
                    value=default,
                    min=lo,
                    max=hi,
                    description=pname.replace("_", " ").title() + ":",
                    style=_slider_style,
                    layout=_slider_layout,
                    **kw,
                )
                self.param_widgets[pname] = slider
                group_sliders.append(slider)
            group_box = VBox(
                [HTML(f"<b style='font-size:0.9em'>{group_label}</b>")] + group_sliders,
                layout=_GROUP_BOX_LAYOUT,
            )
            self.param_group_widgets.append((group_box, required_methods))

        self.no_params_msg = HTML("<p><i>PCA has no adjustable parameters.</i></p>")

        for cb in self.method_checkboxes.values():
            cb.observe(self._on_method_change, names="value")

        self.keep_temp = Checkbox(value=False, description="Keep temporary files")

    def _build_layout(self):
        # Custom annotation grid (hidden by default)
        columns = []
        for category in ANNOTATIONS:
            all_btn, clear_btn = self.group_buttons[category]
            header = HBox([HTML(f"<b>{category}</b>"), all_btn, clear_btn])
            cbs = list(self.annotation_checkboxes[category].values())
            grid = GridBox(
                cbs,
                layout={
                    "grid_template_columns": "repeat(2, auto)",
                    "grid_gap": "0px 8px",
                },
            )
            columns.append(
                VBox(
                    [header, grid],
                    layout={
                        "flex": "1",
                        "min_width": "180px",
                        "border": "1px solid #e0e0e0",
                        "padding": "8px",
                        "margin": "0 4px",
                    },
                )
            )
        self.custom_annotations_box = HBox(columns, layout={"flex_flow": "row wrap"})
        self.custom_annotations_box.layout.display = "none"

        # Parameter grid
        param_grid = GridBox(
            [gw for gw, _ in self.param_group_widgets],
            layout={
                "grid_template_columns": "repeat(auto-fill, minmax(280px, 1fr))",
                "grid_gap": "8px",
                "width": "100%",
            },
        )

        # Advanced options accordion
        advanced_content = VBox(
            [
                HTML("<h4>📋 Annotations</h4>"),
                self.annotation_preset,
                self.custom_annotations_box,
                HTML("<h4>📄 Custom CSV Metadata (optional)</h4>"),
                HTML(
                    "<p><i>Upload a CSV/TSV with per-protein annotations. First column = protein identifiers.</i></p>"
                ),
                self.csv_upload,
                HTML("<h4>📊 Reduction Methods</h4>"),
                HBox(
                    list(self.method_checkboxes.values()),
                    layout={"flex_flow": "row wrap", "gap": "6px"},
                ),
                HTML("<h4>⚙️ Method Parameters</h4>"),
                self.no_params_msg,
                param_grid,
                HTML("<h4>🔧 Options</h4>"),
                self.keep_temp,
            ]
        )

        self.accordion = Accordion(children=[advanced_content])
        self.accordion.set_title(0, "Advanced Options")
        self.accordion.selected_index = None  # collapsed by default

        self.widget = VBox(
            [
                HTML(
                    "<p>Click <b>Generate Bundle</b> to create a ProtSpace visualization file "
                    "using default settings (PCA + UMAP, recommended annotations).</p>"
                ),
                HBox([self.run_button]),
                self.accordion,
                self.output,
            ]
        )

    # ------------------------------------------------------------------
    # Event handlers
    # ------------------------------------------------------------------

    def _on_preset_change(self, change):
        preset = _ANNOTATION_PRESETS.get(change["new"], "default")
        if preset == "custom":
            self.custom_annotations_box.layout.display = ""
        else:
            self.custom_annotations_box.layout.display = "none"
            if preset == "default":
                self._apply_checkbox_preset("default")
            elif preset == "all":
                self._apply_checkbox_preset("all")

    def _on_method_change(self, _change=None):
        for btn in self.method_checkboxes.values():
            btn.button_style = "info" if btn.value else ""
        self._update_param_visibility()

    def _update_param_visibility(self, _change=None):
        selected = {m for m, cb in self.method_checkboxes.items() if cb.value}
        any_params = False
        for group_box, required_methods in self.param_group_widgets:
            visible = bool(required_methods & selected)
            group_box.layout.display = "" if visible else "none"
            if visible:
                any_params = True
        self.no_params_msg.layout.display = "" if not any_params else "none"

    def _select_group(self, category, select_all):
        for cb in self.annotation_checkboxes[category].values():
            cb.value = select_all

    def _apply_checkbox_preset(self, preset):
        for category, checkboxes in self.annotation_checkboxes.items():
            for name, cb in checkboxes.items():
                if preset == "default":
                    cb.value = name in ANNOTATION_DEFAULTS
                elif preset == "all":
                    cb.value = True
                elif preset == "clear":
                    cb.value = False

    def _save_uploaded_csv(self):
        if not self.csv_upload.value:
            return None
        first = list(self.csv_upload.value)[0]
        content = first.get("content", b"")
        if not content:
            return None
        csv_path = Path(first.get("name", "custom_annotations.csv"))
        csv_path.write_bytes(content)
        return str(csv_path)

    def _selected_annotations(self):
        preset = _ANNOTATION_PRESETS.get(self.annotation_preset.value, "default")
        if preset == "none":
            return []
        if preset == "all":
            return [name for names in ANNOTATIONS.values() for name in names]
        if preset == "custom":
            selected = []
            for checkboxes in self.annotation_checkboxes.values():
                for name, cb in checkboxes.items():
                    if cb.value:
                        selected.append(name)
            return selected
        # default
        return list(ANNOTATION_DEFAULTS)

    def _method_commands(self):
        return [
            m.lower().replace("-", "") + "2"
            for m, cb in self.method_checkboxes.items()
            if cb.value
        ]

    # ------------------------------------------------------------------
    # Build & run CLI command
    # ------------------------------------------------------------------

    def _on_run(self, _button):
        with self.output:
            clear_output()

            input_file = globals().get("embedding_file")
            if not input_file or not os.path.exists(str(input_file)):
                print(
                    "❌ No valid embedding file found.\n"
                    "   → Use the Example Datasets cell above, or\n"
                    "   → Upload your own file in the Upload cell."
                )
                return

            input_path = Path(input_file)
            output_path = input_path.parent / (input_path.stem + ".parquetbundle")
            output_path.parent.mkdir(parents=True, exist_ok=True)

            annotations = self._selected_annotations()
            method_cmds = self._method_commands()
            csv_path = self._save_uploaded_csv()

            if not method_cmds:
                print("❌ Select at least one dimensionality reduction method.")
                return

            # Build command
            cmd = [
                "protspace prepare",
                "-i",
                str(input_file),
                "-m",
                ",".join(method_cmds),
                "-o",
                str(output_path),
                "-v",
            ]
            if csv_path:
                cmd.extend(["-a", csv_path])
            if annotations:
                cmd.extend(["-a", ",".join(annotations)])
            for pname, w in self.param_widgets.items():
                cmd.extend([f"--{pname}", str(w.value)])
            if self.keep_temp.value:
                cmd.append("--keep-tmp")

            print("--- ProtSpace Configuration ---")
            print(f"Input:   {input_file}")
            print(f"Output:  {output_path}")
            print(
                f"Methods: {', '.join(m for m, cb in self.method_checkboxes.items() if cb.value)}"
            )
            if csv_path:
                print(f"CSV:     {csv_path}")
            print(f"Annotations: {', '.join(annotations) if annotations else 'None'}")
            print(f"Command: {' '.join(cmd)}\n")

            self._execute(cmd, output_path)

    def _execute(self, cmd, output_path):
        """Run command and render progress bars. Shows all output including errors."""
        progress_widgets = {}
        error_lines = []
        status = HTML(value="<b>🚀 Starting ProtSpace processing...</b>")
        display(status)

        try:
            proc = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
                universal_newlines=True,
            )

            for line in proc.stdout:
                line = line.strip()
                if not line:
                    continue
                if any(p in line for p in _SKIP_PATTERNS):
                    continue

                # Capture error/warning lines
                if line.startswith("ERROR:") or line.startswith("WARNING:"):
                    error_lines.append(line)
                    print(line)
                    continue

                # Update Jupyter progress bars from tqdm lines
                if "Fetching" in line and "%|" in line:
                    for substr, (key, label) in _PROGRESS_TASKS.items():
                        if substr not in line:
                            continue
                        if key not in progress_widgets:
                            bar = IntProgress(
                                value=0,
                                min=0,
                                max=100,
                                bar_style="info",
                                style={"bar_color": "#17a2b8"},
                                layout={"width": "400px"},
                            )
                            lbl = HTML(value=f"<b>{label}</b>")
                            progress_widgets[key] = {"bar": bar, "label": lbl}
                            display(VBox([lbl, bar]))
                        match = re.search(r"(\d+)%", line)
                        if match:
                            pct = int(match.group(1))
                            progress_widgets[key]["bar"].value = pct
                            if pct == 100:
                                progress_widgets[key]["bar"].bar_style = "success"
                                progress_widgets[key][
                                    "label"
                                ].value = f"<b>{label} ✅</b>"
                    continue

                # Show all other output (INFO messages, etc.)
                print(line)

            proc.wait()

            if proc.returncode == 0:
                status.value = "<b>✅ ProtSpace bundle generated successfully!</b>"
                print(f"\nOutput: {output_path}")
                globals()["protspace_output_file"] = str(output_path)
                print("Ready for download — proceed to the next cell.")
            else:
                status.value = (
                    f"<b>❌ Processing failed (exit code {proc.returncode})</b>"
                )
                if error_lines:
                    print("\n--- Error Details ---")
                    for el in error_lines:
                        print(el)
                print("\n--- Troubleshooting ---")
                print(
                    "• Check that the embedding file is a valid HDF5 file with per-protein embeddings"
                )
                print(
                    "• Ensure protein IDs are UniProt accessions (e.g., P12345) for annotation lookup"
                )
                print(
                    "• Try running with 'None' annotations to skip annotation fetching"
                )
                print("• For large datasets (>10k proteins), t-SNE and MDS may be slow")

        except FileNotFoundError:
            print(
                "❌ protspace prepare not found. Re-run the first cell to install ProtSpace."
            )
        except Exception as e:
            print(f"❌ Unexpected error: {e}")


config_widget = ProtSpaceConfigWidget()
display(config_widget.widget)

In [ ]:
# @title 📥 Download ProtSpace Bundle {display-mode: "form"}
# @markdown Run this cell, then upload the downloaded file at https://protspace.app/

if "protspace_output_file" not in globals():
    print("❌ No output file found. Run the generation cell first.")
else:
    output_path = Path(globals()["protspace_output_file"])
    if output_path.exists():
        size_mb = output_path.stat().st_size / (1024 * 1024)
        print(f"✅ {output_path.name} ({size_mb:.2f} MB)")
        print("📦 Starting download...\n")
        try:
            files.download(str(output_path))
            print("✅ Download complete!")
            print(f"🌐 Next: upload {output_path.name} at https://protspace.app/")
        except Exception as e:
            print(f"❌ Download failed: {e}")
    else:
        print(f"❌ Output file not found: {output_path}")